# 03 — Recherche et RAG

Recherche lexicale TF-IDF avec similarité cosinus sur le train 2025. Le mode par défaut retourne des extraits avec leurs sources ; il ne réalise pas de génération. Une synthèse par modèle local Ollama peut être activée explicitement. Aucun embedding ou modèle externe n’est téléchargé par ce notebook.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Ouvrir depuis la racine ou notebooks/'
sys.path.insert(0, str(ROOT))
import json
import sqlite3
from contextlib import closing
import pandas as pd
REPORTS = ROOT / 'reports/demo'
DATABASE = ROOT / 'data/processed/demo.sqlite'
assert DATABASE.is_file(), 'Executer python -m scripts.prepare_demo depuis la racine'


In [2]:
import joblib
from src.retriever import search
from src.rag import ask
index = joblib.load(ROOT / 'artifacts/demo/retriever.joblib')
question = 'credit report identity theft'
# Comparer k=3 et k=5 : ces scores sont des similarites, pas des probabilites.
experiments = []
for k in [3, 5]:
    for source in search(index, question, k):
        experiments.append({'k': k, **{field: source[field] for field in ['complaint_id', 'source_file', 'score']}})
pd.DataFrame(experiments)

,k,complaint_id,source_file,score
0,3,13293607,CCDB_Export_11_May_2025_through_June_2025.csv,0.314390
1,3,15651725,CCDB_Export_13_September_2025_through_October_...,0.253114
2,3,13260546,CCDB_Export_11_May_2025_through_June_2025.csv,0.238306
3,5,13293607,CCDB_Export_11_May_2025_through_June_2025.csv,0.314390
4,5,15651725,CCDB_Export_13_September_2025_through_October_...,0.253114
5,5,13260546,CCDB_Export_11_May_2025_through_June_2025.csv,0.238306
6,5,12239958,CCDB_Export_10_March_2025_through_April_2025.csv,0.223788
7,5,11599301,CCDB_Export_9_January_2025_through_February_20...,0.223534


In [3]:
answer = ask(index, question)
# Apercu bref; les extraits complets sont consultables avec la CLI.
print(answer['mode'])
print(answer['answer'][:240])
print('Sources :', [source['complaint_id'] for source in answer['sources']])

extractive
[13293607] I am writing to urgently request the removal of fraudulent accounts on my credit report due to identity theft. The inaccurate information is causing significant harm to my credit, and I expect immediate corrective action.

[15651
Sources : ['13293607', '15651725', '13260546']


In [4]:
ask(index, 'quasar nebula astrophysics')

{'mode': 'abstention',
 'answer': 'Aucun passage pertinent trouve dans cet index.',
 'sources': [],
 'citations_valid': True}

## Génération locale facultative
Après installation d’Ollama et d’un modèle compatible, définir `OLLAMA_MODEL` dans le terminal puis appeler `python -m src.demo ask "credit report identity theft" --artifacts artifacts/demo --generate`. La requête envoyée au modèle contient la question et les extraits sourcés. Le contrôle des IDs cités ne garantit pas la fidélité des affirmations. La génération Ollama n’est pas exécutée ni évaluée ici. [Documentation du protocole](https://docs.ollama.com/api/chat).

L’abstention est déclenchée uniquement en l’absence de recouvrement lexical. Une question hors domaine partageant des mots du corpus peut donc produire des résultats. Comparer k=3 et k=5 ne mesure pas leur pertinence : il faut des jugements humains indépendants.